In [19]:
from qdrant_client import QdrantClient
client = QdrantClient(url="http://localhost:6333")

In [20]:
from qdrant_client.models import Distance, VectorParams


client.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=4, distance=Distance.DOT),
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `test_collection` already exists!"},"time":0.000101511}'

In [21]:
from qdrant_client.models import PointStruct

operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London"}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow"}),
        PointStruct(id=4, vector=[0.18, 0.01, 0.85, 0.80], payload={"city": "New York"}),
        PointStruct(id=5, vector=[0.24, 0.18, 0.22, 0.44], payload={"city": "Beijing"}),
        PointStruct(id=6, vector=[0.35, 0.08, 0.11, 0.44], payload={"city": "Mumbai"}),
    ],
)

print(operation_info)

operation_id=3 status=<UpdateStatus.COMPLETED: 'completed'>


In [22]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

search_result = client.query_points(
    collection_name="test_collection",
    query=[0.2, 0.1, 0.9, 0.7],
    query_filter=Filter(
        must=[FieldCondition(key="city", match=MatchValue(value="London"))]
    ),
    with_payload=True,
    limit=3,
).points

print(search_result)

AttributeError: 'QdrantClient' object has no attribute 'query_points'

In [23]:
import ollama

documents = [
    "Ollama is a powerful tool for running LLMs locally.",
    "Text embeddings are numerical representations of text.",
    "The quick brown fox jumps over the lazy dog."
]

response = ollama.embed(model='nomic-embed-text', input=documents)

# This will be a list of embeddings
embeddings = [r for r in response['embeddings']]

print(f"Generated {len(embeddings)} embeddings.")
print(f"Dimensions of first embedding: {len(embeddings[0])}")

Generated 3 embeddings.
Dimensions of first embedding: 768


In [24]:
import json 
with open('product.json', 'r') as f:
    products = json.load(f)
products[0]


{'product_id': 'PROD101',
 'title': 'Elegant Floral Print Midi Dress',
 'description': 'Beautiful midi dress featuring a stunning floral print on premium quality fabric. This versatile piece offers a flattering A-line silhouette with a cinched waist and flowing skirt. Perfect for both casual outings and semi-formal events. The dress includes adjustable shoulder straps, hidden side zipper, and a lined interior for comfort. Made from breathable cotton-blend material that maintains its shape and color even after multiple washes. The dress falls just below the knee, making it appropriate for various occasions.',
 'price': 79.99,
 'size': ['XS', 'S', 'M', 'L', 'XL'],
 'color': ['Navy Blue', 'Rose Pink', 'Mint Green'],
 'gender': 'Female',
 'category': 'Dresses',
 'brand': 'StyleHaven',
 'tags': ['floral',
  'midi',
  'casual',
  'formal',
  'summer',
  'versatile',
  'cotton-blend']}

In [25]:
import tqdm 
for product in tqdm.tqdm(products):
    string_data = {
        'title': product['title'],
        'description': product['description'],
        'color': product['color'],
        'category': product['category'],
        'brand' : product['brand'],
        'tags' : " ".join(product['tags'])
    }
    product_id = int(product['product_id'].replace('PROD', ''))
    payload = {
        'title': product['title'],
        'price': product['price'],
        'size': product['size'],
        'color': product['color'],
        'gender': product['gender'],
        'category': product['category'],
        'brand' : product['brand'],
        'tags' : product['tags']
    }

    string_data = json.dumps(string_data)
    response = ollama.embed(model='nomic-embed-text', input=string_data)
    operation_info = client.upsert( 
        collection_name="products",
        wait=True,
        points=[
            PointStruct(id=product_id, vector=response['embeddings'][0], payload=payload),
        ],
)

100%|██████████| 84/84 [00:09<00:00,  9.01it/s]


In [32]:
from qdrant_client.http.models import Filter, FieldCondition, MatchValue, MatchAny

question = "Looking for jeans"
response = ollama.embed(model='nomic-embed-text', input=question)

search_result = client.search(
    collection_name="products",
    query_vector=response['embeddings'][0],
    query_filter=Filter(
        must=[
            FieldCondition(key="gender", match=MatchValue(value="Male")),
        ]
    ),
    with_payload=True,
    limit=10
)

for result in search_result:
    print(f"Product ID: {result.id}")
    print(f"Score: {result.score}")
    print(f"Payload: {result.payload}")
    print("-----")

Product ID: 155
Score: 0.6717948
Payload: {'title': 'Slim Fit Stretch Jeans - Light Blue', 'price': 64.99, 'size': ['28', '30', '32', '34', '36', '38'], 'color': ['Light Blue', 'Medium Wash'], 'gender': 'Male', 'category': 'Bottoms', 'brand': 'UrbanDenim', 'tags': ['jeans', 'slim-fit', 'stretch', 'casual', 'everyday', 'light-wash']}
-----
Product ID: 115
Score: 0.66749644
Payload: {'title': 'Classic Fit Denim Jeans - Dark Wash', 'price': 69.99, 'size': ['28', '30', '32', '34', '36', '38', '40'], 'color': ['Dark Indigo'], 'gender': 'Male', 'category': 'Bottoms', 'brand': 'DenimCraft', 'tags': ['jeans', 'denim', 'classic-fit', 'casual', 'everyday', 'dark-wash']}
-----
Product ID: 240
Score: 0.53714585
Payload: {'title': 'Tech-Friendly Chino Pants', 'price': 79.99, 'size': ['30', '32', '34', '36', '38', '40'], 'color': ['Khaki', 'Navy', 'Grey', 'Olive'], 'gender': 'Male', 'category': 'Bottoms', 'brand': 'TechWear', 'tags': ['chinos', 'wrinkle-resistant', 'business-casual', 'tech', 'travel

In [9]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="products",
    vectors_config=VectorParams(size=768, distance=Distance.DOT),
)

True

In [11]:
operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
    ],
)